# 1. Config e imports

In [1]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.8/31.8 MB 62.2 MB/s eta 0:00:00


In [2]:
import os
import time
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from sklearn.metrics import f1_score
import gc
from torch.cuda.amp import autocast, GradScaler

# Configuración de dispositivo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Utilizando dispositivo: {device}")

# Rutas de entrada (Asegúrate de haber añadido el dataset con los folds de la Fase 1)
TRAIN_FOLDS_PATH = "/kaggle/input/datasets/ernestomiguez/casmi2026-phase1-folds/train_folds.parquet"
TEST_PATH = "/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/test.parquet"
TRAIN_PATH = "/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/train.parquet"

Utilizando dispositivo: cuda


# 2. Funciones de Limpieza de Espectros MS2

In [3]:
def clean_spectrum(mzs, intensities, precursor_mz, top_n=128, min_rel_intensity=0.01):
    """
    Limpia y filtra un espectro MS2 según las reglas recomendadas de la competición:
    1. Elimina picos > precursor_mz + 2.0 Da.
    2. Elimina picos con intensidad relativa por debajo del min_rel_intensity (1%).
    3. Mantiene los top_n picos más intensos.
    4. Aplica transformación de raíz cuadrada a las intensidades.
    """
    mzs = np.array(mzs, dtype=np.float32)
    intensities = np.array(intensities, dtype=np.float32)
    
    # 1. Filtro de masa superior al precursor
    mask_mass = mzs <= (precursor_mz + 2.0)
    mzs, intensities = mzs[mask_mass], intensities[mask_mass]
    
    if len(mzs) == 0:
        return np.array([0.0], dtype=np.float32), np.array([0.0], dtype=np.float32)
    
    # 2. Re-normalizar al pico base y filtrar por intensidad relativa mínima
    max_intensity = intensities.max()
    if max_intensity > 0:
        rel_intensities = intensities / max_intensity
    else:
        rel_intensities = intensities
        
    mask_int = rel_intensities >= min_rel_intensity
    mzs, intensities = mzs[mask_int], rel_intensities[mask_int]
    
    # 3. Retener Top-N picos
    if len(mzs) > top_n:
        top_indices = np.argsort(intensities)[-top_n:]
        mzs, intensities = mzs[top_indices], intensities[top_indices]
        # Reordenar por m/z ascendente
        sort_mz_idx = np.argsort(mzs)
        mzs, intensities = mzs[sort_mz_idx], intensities[sort_mz_idx]
        
    # 4. Transformación Square-Root para suavizar picos dominantes
    intensities = np.sqrt(intensities)
    
    return mzs, intensities

def spectrum_to_binned_vector(mzs, intensities, bin_size=0.1, max_mz=1200.0):
    """
    Convierte la lista de picos (m/z, intensidad) en un vector denso por bining.
    """
    num_bins = int(max_mz / bin_size)
    vector = np.zeros(num_bins, dtype=np.float32)
    
    indices = (mzs / bin_size).astype(int)
    valid_mask = (indices >= 0) & (indices < num_bins)
    
    for idx, int_val in zip(indices[valid_mask], intensities[valid_mask]):
        vector[idx] = max(vector[idx], int_val)  # O acumulación / max
        
    return vector

# 3. Generación de Fingerprints (Targets) a partir de SMILES

In [4]:
# Instanciamos el generador de Morgan Fingerprints fuera para reutilizarlo
# radius=2 (equivalente a ECFP4), fpSize=1024 bits
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)

def smiles_to_morgan_fingerprint(smiles, n_bits=1024):
    """
    Genera un Morgan Fingerprint moderno (ECFP4) utilizando rdFingerprintGenerator.
    """
    if not isinstance(smiles, str) or not smiles.strip():
        return np.zeros(n_bits, dtype=np.float32)
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return np.zeros(n_bits, dtype=np.float32)
        
        # Generar fingerprint con la API moderna de RDKit
        fp = morgan_gen.GetFingerprint(mol)
        arr = np.zeros((n_bits,), dtype=np.uint8)
        Chem.DataStructs.ConvertToNumpyArray(fp, arr)
        return arr.astype(np.float32)
    except Exception:
        return np.zeros(n_bits, dtype=np.float32)

# 4. Dataset y Modelo PyTorch para Predicción de Fingerprints

In [5]:
class MS2Dataset(Dataset):
    def __init__(self, df, bin_size=0.5, max_mz=1000.0, is_train=True):
        self.df = df.reset_index(drop=True)
        self.bin_size = bin_size
        self.max_mz = max_mz
        self.num_bins = int(max_mz / bin_size)
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Limpieza de espectro
        clean_mzs, clean_ints = clean_spectrum(
            row['ms2_mzs'], 
            row['ms2_normalized_intensities'], 
            row['precursor_mz']
        )
        
        # Binned vector
        spec_vec = spectrum_to_binned_vector(clean_mzs, clean_ints, self.bin_size, self.max_mz)
        
        # Añadir precursor_mz como feature escalar al final del vector
        x = np.append(spec_vec, np.float32(row['precursor_mz'] / 1000.0))
        
        if self.is_train:
            y = smiles_to_morgan_fingerprint(row['normalized_smiles'], n_bits=1024)
            return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)
        else:
            return torch.tensor(x, dtype=torch.float32)

class ResidualBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.ReLU(),
            nn.Dropout(0.3), # Dropout moderado para evitar sobreajuste rápido
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim)
        )
        self.relu = nn.ReLU()

    def forward(self, x):
        # Conexión residual: la entrada se suma a la salida del bloque
        return self.relu(x + self.block(x))

class EfficientResidualSpectrumMLP(nn.Module):
    def __init__(self, input_dim=2001, hidden_dim=1024, output_dim=1024, num_blocks=2):
        super().__init__()
        # Capa de entrada para proyectar al espacio oculto
        self.input_layer = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU()
        )
        
        # Bloques residuales apilados (eficientes en tiempo de cómputo)
        self.res_blocks = nn.ModuleList([ResidualBlock(hidden_dim) for _ in range(num_blocks)])
        
        # Capa de salida hacia los 1024 bits del fingerprint
        self.output_layer = nn.Sequential(
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        x = self.input_layer(x)
        for block in self.res_blocks:
            x = block(x)
        return self.output_layer(x)

# 5. Carga de Folds y Preparación de DataLoaders

In [6]:
print("Cargando dataset original y folds asignados...")
full_train_df = pd.read_parquet(TRAIN_PATH)
folds_df = pd.read_parquet(TRAIN_FOLDS_PATH)

if 'fold' in full_train_df.columns:
    full_train_df = full_train_df.drop(columns=['fold'])

train_folds_df = full_train_df.merge(
    folds_df[['normalized_smiles', 'inchikey14', 'fold']].drop_duplicates(),
    on=['normalized_smiles', 'inchikey14'],
    how='inner'
)

# Separar en Train y Validation usando el Fold 0
train_df = train_folds_df[train_folds_df['fold'] != 0].reset_index(drop=True)
val_df = train_folds_df[train_folds_df['fold'] == 0].reset_index(drop=True)

# Liberar memoria de los DataFrames pesados que ya no se usan
del full_train_df, train_folds_df
import gc; gc.collect()

print(f"Espectros para Entrenamiento: {len(train_df):,}")
print(f"Espectros para Validación:    {len(val_df):,}")

# Crear PyTorch Datasets
train_dataset = MS2Dataset(train_df, bin_size=0.5, max_mz=1000.0, is_train=True)
val_dataset = MS2Dataset(val_df, bin_size=0.5, max_mz=1000.0, is_train=True)

# DataLoaders optimizados para evitar RAM OOM
train_loader = DataLoader(
    train_dataset, 
    batch_size=128,        # Mantener en 128 para prevenir GPU VRAM OOM
    shuffle=True, 
    num_workers=2,         # Máximo 2 workers para la RAM de Kaggle
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=128, 
    shuffle=False, 
    num_workers=2, 
    pin_memory=True
)

print("✓ DataLoaders configurados para prevenir OOM.")

Cargando dataset original y folds asignados...
Espectros para Entrenamiento: 2,031,686
Espectros para Validación:    507,922
✓ DataLoaders configurados para prevenir OOM.


# 6. Instanciación del Modelo, Función de Pérdida y Optimizador

In [7]:
# Dimensión de entrada: num_bins (1000.0 / 0.5 = 2000) + 1 (precursor_mz)
INPUT_DIM = int(1000.0 / 0.5) + 1
OUTPUT_DIM = 1024  # Tamaño del Morgan Fingerprint

model = EfficientResidualSpectrumMLP(input_dim=2001, hidden_dim=1024, output_dim=1024, num_blocks=2).to(device)

class CombinedLossWithLogits(nn.Module):
    def __init__(self, bce_weight=0.5, cosine_weight=0.5):
        super().__init__()
        # Usamos BCEWithLogitsLoss (seguro para autocast)
        self.bce_with_logits = nn.BCEWithLogitsLoss()
        self.bce_weight = bce_weight
        self.cosine_weight = cosine_weight
        
    def forward(self, logits, targets):
        # 1. Pérdida BCE directa sobre los logits
        loss_bce = self.bce_with_logits(logits, targets)
        
        # 2. Para el coseno, aplicamos sigmoide a los logits para obtener probabilidades entre 0 y 1
        preds = torch.sigmoid(logits)
        
        dot_product = (preds * targets).sum(dim=1)
        pred_norm = preds.norm(dim=1, p=2)
        target_norm = targets.norm(dim=1, p=2)
        
        cosine_sim = dot_product / (pred_norm * target_norm + 1e-8)
        loss_cosine = (1.0 - cosine_sim).mean()
        
        return (self.bce_weight * loss_bce) + (self.cosine_weight * loss_cosine)

# 7. Bucle de entrenamiento

In [8]:
# --- 3. BUCLE DE ENTRENAMIENTO CON EARLY STOPPING ---
def train_model_lightweight(model, train_loader, val_loader, optimizer, criterion, device, epochs=15, patience=3):
    best_val_loss = float('inf')
    best_model_wts = copy.deepcopy(model.state_dict())
    patience_counter = 0
    
    # Sintaxis actualizada para evitar warnings y consumo innecesario
    scaler = torch.amp.GradScaler('cuda')

    print("Iniciando entrenamiento ligero antifugas de memoria...")
    
    for epoch in range(epochs):
        epoch_start_time = time.time()
        
        gc.collect()
        torch.cuda.empty_cache()
        
        # --- Fase de Entrenamiento ---
        model.train()
        train_loss = 0.0
        for x_batch, y_batch in train_loader:
            x_batch = x_batch.to(device, non_blocking=True)
            y_batch = y_batch.to(device, non_blocking=True)
            
            optimizer.zero_grad(set_to_none=True)
            
            with torch.amp.autocast('cuda'):
                logits = model(x_batch)
                loss = criterion(logits, y_batch)
                
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            train_loss += loss.item() * x_batch.size(0)
            
        epoch_train_loss = train_loss / len(train_loader.dataset)
        
        # --- Fase de Validación (Optimizada para no saturar RAM) ---
        model.eval()
        val_loss = 0.0
        all_val_preds = []
        all_val_targets = []
        
        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                x_batch = x_batch.to(device, non_blocking=True)
                y_batch = y_batch.to(device, non_blocking=True)
                
                with torch.amp.autocast('cuda'):
                    logits = model(x_batch)
                    loss = criterion(logits, y_batch)
                    
                val_loss += loss.item() * x_batch.size(0)
                
                # Convertir a CPU y guardar solo en formato compacto para evitar fugas de memoria
                preds_bin = (torch.sigmoid(logits).detach().cpu().numpy() > 0.5).astype(np.int8)
                targets_np = y_batch.detach().cpu().numpy().astype(np.int8)
                
                all_val_preds.append(preds_bin)
                all_val_targets.append(targets_np)
                
                # Liberar tensores locales inmediatamente
                del logits, preds_bin, targets_np
                
        epoch_val_loss = val_loss / len(val_loader.dataset)
        
        # Concatenar de forma segura
        all_preds = np.concatenate(all_val_preds, axis=0)
        all_targets = np.concatenate(all_val_targets, axis=0)
        val_f1 = f1_score(all_targets, all_preds, average='micro', zero_division=0)
        
        # Limpieza estricta de variables temporales
        del all_val_preds, all_val_targets, all_preds, all_targets
        torch.cuda.empty_cache()
        gc.collect()
        
        elapsed_time = time.time() - epoch_start_time
        print(f"Epoch [{epoch+1:02d}/{epochs:02d}] ({elapsed_time:.1f}s) | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val F1: {val_f1:.4f}")
        
        # --- Early Stopping ---
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            patience_counter = 0
            print(f"  --> Guardado nuevo mejor modelo (Val Loss: {best_val_loss:.4f})")
        else:
            patience_counter += 1
            print(f"  --> Sin mejora en val loss ({patience_counter}/{patience})")
            if patience_counter >= patience:
                print(f"\nEarly Stopping activado en la época {epoch+1}.")
                break

    model.load_state_dict(best_model_wts)
    return model

# --- 4. CONFIGURACIÓN E INICIALIZACIÓN ---

# Aumentamos el batch_size a 256 para acelerar el procesamiento en GPU
BATCH_SIZE = 64

# Supongamos que train_loader y val_loader ya están definidos en tu notebook con MS2Dataset
# train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
# val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

model = EfficientResidualSpectrumMLP(input_dim=2001, hidden_dim=1024, output_dim=1024, num_blocks=2).to(device)
criterion = CombinedLossWithLogits(bce_weight=0.5, cosine_weight=0.5)

# Optimizador con mayor weight_decay para frenar el overfitting
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)

# Ejecutar entrenamiento inteligente
trained_model = train_model_lightweight(model, train_loader, val_loader, optimizer, criterion, device, epochs=15, patience=3)

torch.save(trained_model.state_dict(), 'fingerprint_mlp_fold0.pth')

print("\n¡Entrenamiento finalizado exitosamente!")

Iniciando entrenamiento ligero antifugas de memoria...
Epoch [01/15] (1212.6s) | Train Loss: 0.2841 | Val Loss: 0.2914 | Val F1: 0.4191
  --> Guardado nuevo mejor modelo (Val Loss: 0.2914)
Epoch [02/15] (1147.2s) | Train Loss: 0.2639 | Val Loss: 0.2884 | Val F1: 0.4315
  --> Guardado nuevo mejor modelo (Val Loss: 0.2884)
Epoch [03/15] (1142.9s) | Train Loss: 0.2555 | Val Loss: 0.2872 | Val F1: 0.4359
  --> Guardado nuevo mejor modelo (Val Loss: 0.2872)
Epoch [04/15] (1167.0s) | Train Loss: 0.2504 | Val Loss: 0.2870 | Val F1: 0.4386
  --> Guardado nuevo mejor modelo (Val Loss: 0.2870)
Epoch [05/15] (1165.6s) | Train Loss: 0.2468 | Val Loss: 0.2861 | Val F1: 0.4404
  --> Guardado nuevo mejor modelo (Val Loss: 0.2861)
Epoch [06/15] (1039.5s) | Train Loss: 0.2441 | Val Loss: 0.2857 | Val F1: 0.4432
  --> Guardado nuevo mejor modelo (Val Loss: 0.2857)
Epoch [07/15] (1034.8s) | Train Loss: 0.2420 | Val Loss: 0.2857 | Val F1: 0.4462
  --> Sin mejora en val loss (1/3)
Epoch [08/15] (1033.1s) |

# 8. Guardar los Embeddings / Fingerprints predichos

In [9]:
# Cargar los mejores pesos guardados
model.load_state_dict(torch.load('fingerprint_mlp_fold0.pth'))
model.eval()

# Cargar el test.parquet
test_df = pd.read_parquet(TEST_PATH)
test_dataset = MS2Dataset(test_df, bin_size=0.5, max_mz=1000.0, is_train=False)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False, num_workers=2)

test_preds = []
with torch.no_grad():
    for x_batch in test_loader:
        x_batch = x_batch.to(device)
        preds = model(x_batch)
        test_preds.append(preds.cpu().numpy())

test_preds_arr = np.concatenate(test_preds, axis=0)

# Guardar las predicciones de fingerprints para usarlas en la Fase 3
np.save('test_predicted_fingerprints.npy', test_preds_arr)
print("✓ Se han guardado 'fingerprint_mlp_fold0.pth' y 'test_predicted_fingerprints.npy'.")

✓ Se han guardado 'fingerprint_mlp_fold0.pth' y 'test_predicted_fingerprints.npy'.
